# markdown_it_py: 5 Basic Usage Examples

In [ ]:
# Example 1: Basic markdown parsing
from markdown_it import MarkdownIt
md = MarkdownIt()
result = md.render('# Hello World\n\nThis is **bold** and *italic* text.')
print(result)
assert result == '<h1>Hello World</h1>\n<p>This is <strong>bold</strong> and <em>italic</em> text.</p>\n', result


In [ ]:
# Example 2: Options (hard line breaks)
# 'breaks' turns a newline inside a paragraph into <br />; without it the newline stays a soft break.
# (Turning URLs into links is the separate 'linkify' rule, which needs the optional linkify-it-py package.)
md = MarkdownIt('commonmark', {'breaks': True})
text = 'Check out https://www.python.org\nNew line here.'
result = md.render(text)
print(result)
assert result == '<p>Check out https://www.python.org<br />\nNew line here.</p>\n', result
assert MarkdownIt('commonmark').render(text) == '<p>Check out https://www.python.org\nNew line here.</p>\n'


In [ ]:
# Example 3: Parse tokens
tokens = md.parse('## Heading\n\n- List item 1\n- List item 2')
for token in tokens:
    print(f'{token.type}: {token.tag} - {token.content}')
assert [t.type for t in tokens][:3] == ['heading_open', 'inline', 'heading_close'], [t.type for t in tokens]
assert tokens[0].tag == 'h2' and tokens[3].type == 'bullet_list_open', (tokens[0].tag, tokens[3].type)
assert [t.content for t in tokens if t.type == 'inline'] == ['Heading', 'List item 1', 'List item 2']


In [ ]:
# Example 4: Custom renderer
from markdown_it.common.utils import escapeHtml
from markdown_it.renderer import RendererHTML

class InlineCodeRenderer(RendererHTML):
    """Renderer rules are methods named after token types; this one overrides inline code."""
    def code_inline(self, tokens, idx, options, env):
        return f'<code class="inline">{escapeHtml(tokens[idx].content)}</code>'

md = MarkdownIt(renderer_cls=InlineCodeRenderer)
result = md.render('> This is a blockquote\n\n`code <here>`')
print(result)
assert result == '<blockquote>\n<p>This is a blockquote</p>\n</blockquote>\n<p><code class="inline">code &lt;here&gt;</code></p>\n', result


In [ ]:
# Example 5: Environment and options
md = MarkdownIt()
env = {}
tokens = md.parse('# Title\n\nParagraph with [link][docs]\n\n[docs]: https://example.com "Docs"', env)
print(f'Environment: {env}')
print(f'Token count: {len(tokens)}')
# parse() collects reference definitions into env; the link token resolves to the referenced URL
assert env['references']['DOCS']['href'] == 'https://example.com', env
assert len(tokens) == 6, [t.type for t in tokens]
link = next(child for child in tokens[4].children if child.type == 'link_open')
assert link.attrs == {'href': 'https://example.com', 'title': 'Docs'}, link.attrs
